# Real ClinicalBERT fine-tune

Fine-tunes `emilyalsentzer/Bio_ClinicalBERT` (Alsentzer et al., 2019) for 4-class risk classification (NONE/LOW/MEDIUM/HIGH), exactly as specified in the project plan.

**Before running:** Runtime -> Change runtime type -> T4 GPU (free tier).

**Upload these three files first** (left sidebar -> Files -> upload):
- `clinicalbert_train.csv`
- `clinicalbert_test.csv`
- `clinicalbert_real_validation.csv`

(Generated locally by `data/export_for_clinicalbert.py` -- same sentences the TF-IDF substitute was trained on, so results are directly comparable.)

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (fine-tuning will be slow -- go to Runtime > Change runtime type > T4 GPU)')

## Load data

In [ ]:
train_df = pd.read_csv('clinicalbert_train.csv')
test_df = pd.read_csv('clinicalbert_test.csv')
real_df = pd.read_csv('clinicalbert_real_validation.csv')

print('Train:', len(train_df), train_df['label'].value_counts().to_dict())
print('Test: ', len(test_df), test_df['label'].value_counts().to_dict())
print('Real: ', len(real_df), real_df['label'].value_counts().to_dict())

train_df.head()

In [ ]:
LABELS = ['NONE', 'LOW', 'MEDIUM', 'HIGH']
le = LabelEncoder()
le.fit(LABELS)

train_df['label_id'] = le.transform(train_df['label'])
test_df['label_id'] = le.transform(test_df['label'])
real_df['label_id'] = le.transform(real_df['label'])

id2label = {i: l for i, l in enumerate(le.classes_)}
label2id = {l: i for i, l in enumerate(le.classes_)}
print(id2label)

## Load Bio_ClinicalBERT and tokenize

In [ ]:
MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=4, id2label=id2label, label2id=label2id
)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=64)

train_ds = Dataset.from_pandas(train_df[['text', 'label_id']].rename(columns={'label_id': 'label'}))
test_ds = Dataset.from_pandas(test_df[['text', 'label_id']].rename(columns={'label_id': 'label'}))
real_ds = Dataset.from_pandas(real_df[['text', 'label_id']].rename(columns={'label_id': 'label'}))

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
real_ds = real_ds.map(tokenize, batched=True)

## Fine-tune (approx. 2-4 minutes on a free T4 GPU for a dataset this size)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

args = TrainingArguments(
    output_dir='./clinicalbert_output',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=10,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

## Evaluate on synthetic test set (same split the TF-IDF model used)

In [ ]:
test_pred = trainer.predict(test_ds)
test_preds = np.argmax(test_pred.predictions, axis=1)
test_true = test_df['label_id'].values

acc = accuracy_score(test_true, test_preds)
print(f'ClinicalBERT accuracy on synthetic test set: {acc:.4f}')
print()
print(classification_report(test_true, test_preds, target_names=LABELS, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(test_true, test_preds))
print()
print('COMPARE THIS to your TF-IDF substitute result: 98.7% accuracy (see backend/risk_models/evaluate.py output)')

## Evaluate on REAL Synthea validation set (external validation, same as the TF-IDF model was tested on)

In [ ]:
real_pred = trainer.predict(real_ds)
real_preds = np.argmax(real_pred.predictions, axis=1)
real_true = real_df['label_id'].values

acc_real = accuracy_score(real_true, real_preds)
print(f'ClinicalBERT accuracy on REAL Synthea drug-switch cases: {acc_real:.4f} ({int(acc_real*len(real_df))}/{len(real_df)})')
print()
print('COMPARE THIS to your TF-IDF substitute result: 90.0% (27/30) -- see backend/risk_models/evaluate_real_synthea.py output')
print('And to Random Forest: 96.7% (29/30)')

## Save the fine-tuned model and results (download these before your Colab session ends)

In [ ]:
import json

trainer.save_model('./clinicalbert_finetuned')
tokenizer.save_pretrained('./clinicalbert_finetuned')

results = {
    'model': MODEL_NAME,
    'synthetic_test_accuracy': float(acc),
    'real_synthea_accuracy': float(acc_real),
    'real_synthea_correct': int(acc_real * len(real_df)),
    'real_synthea_total': len(real_df),
}
with open('clinicalbert_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

!zip -r clinicalbert_finetuned.zip clinicalbert_finetuned clinicalbert_results.json

from google.colab import files
files.download('clinicalbert_finetuned.zip')

## What to bring back to the project

1. `clinicalbert_results.json` -- the three headline numbers to report
2. The printed classification reports and confusion matrices above (screenshot or copy into your evaluation write-up)
3. `clinicalbert_finetuned.zip` if you want to actually serve this model instead of the TF-IDF substitute (would need `transformers` + PyTorch added to `backend/requirements.txt` and a new inference script -- ask if you want help wiring that in)